In [63]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
from pathlib import Path

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

from sklearn.preprocessing import LabelEncoder


In [64]:
dir = Path('/kaggle/input/nfl-big-data-bowl-2026-prediction')
trial = False


def read():
    train_in= pd.read_csv(dir / 'train/input_2023_w01.csv')
    train_out = pd.read_csv(dir / 'train/output_2023_w01.csv')
    test_in= pd.read_csv(dir / 'test_input.csv')
    test_out = pd.read_csv(dir / 'test.csv')
    
    if trial:
        mask = (train_in['player_to_predict']==True) | (train_in['game_id']==2023090700)
        train_in = train_in[mask]
        return train_in, train_out, test_in, test_out
    for i in range(2, 19):
        train_in = pd.concat([train_in, pd.read_csv(dir / 'train/input_2023_w{}.csv'.format('0'+str(i) if len(str(i))==1 else i))], axis=0)
        train_out = pd.concat([train_out, pd.read_csv(dir / 'train/output_2023_w{}.csv'.format('0'+str(i) if len(str(i))==1 else i))], axis=0)
    return train_in, train_out, test_in, test_out



In [65]:
def del_cols(df_in_, df_out_):
    df_in = df_in_.copy()
    df_out = df_out_.copy()
    df_in = df_in[input_cols]
    if 'x' in df_out.columns:
        df_out = df_out[output_cols]
    else:
        df_out = df_out[output_cols_]
    return df_in, df_out
    

def LE_train(df_):
    df = df_.copy()
    encode_list = {}
    for col in df.select_dtypes(exclude=['Int64', 'Float64', 'Int32', 'int64']).columns:
        # if col=='play_direction': continue
        unique = df[col].unique()
        encoder = {u:i for i, u in enumerate(unique)}
        df[col] = df[col].map(encoder)
        encode_list[col] = encoder
    return df, encode_list

def LE_test(df_, encode_list):
    df = df_.copy()
    for col in df.select_dtypes(exclude=['Int64', 'Float64', 'Int32', 'int64']).columns:
        df[col] = df[col].map(encode_list[col])
        df[col] = df[col].fillna(-1).astype(int)
    return df

def f(df_in_, df_out_, encode_list=None):
    df_in = df_in_.copy()
    df_out = df_out_.copy()

    pbd = 'player_birth_date'
    if pbd in df_in.columns:
        df_in[pbd] = pd.to_datetime(df_in[pbd])
        year = df_in[pbd].dt.year
        month = df_in[pbd].dt.month
        day = df_in[pbd].dt.day
        df_in[pbd] = year*400 + month*31 + day

    ph = 'player_height'
    if ph in df_in.columns:
        left_right = df_in[ph].str.split('-', expand=True)
        df_in[ph] = left_right[0].astype(int) * 12 + left_right[1].astype(int)
        
    if encode_list:
        df_in_2 = LE_test(df_in, encode_list)
    else:
        df_in_2, encode_list = LE_train(df_in)

    return df_in_2, df_out, encode_list


    

In [66]:
trace_num = 6

def get_all_data(df_in, df_out):    
    ids = {}
    x = []
    y = []
    id_to_values = []
    idx = 0
    
    rows = []
    
    x_idx = list(df_in.columns).index('x')
    y_idx = list(df_in.columns).index('y')
    x_ball_idx = list(df_in.columns).index('ball_land_x')
    y_ball_idx = list(df_in.columns).index('ball_land_y')
    
    for row in df_in.itertuples(index=False, name=None):
        t = (row[0], row[1], row[3])
        if t in ids:
            j = ids[t]
        else:
            ids[t] = idx
            j = idx
            idx += 1
            id_to_values.append(row)
            x.append([])
            y.append([])
            
        if not x[j]:
            x[j].append(row[x_idx])
            y[j].append(row[y_idx])
            continue
    
        values = list(id_to_values[j])
        values[x_idx] = x[j][-1]
        values[y_idx] = y[j][-1]
        x[j].append(row[x_idx])
        y[j].append(row[y_idx])
    
        if len(x[j])<trace_num+1: 
            continue

        
        values.append(values[x_ball_idx]-values[x_idx])
        values.append(values[y_ball_idx]-values[y_idx])
        
        for k in range(min(trace_num, len(x[j])-1)):
            values.append(x[j][~k]-x[j][~(k+1)])
            values.append(y[j][~k]-y[j][~(k+1)])
        rows.append(values)




    
    x_idx_out = list(df_out.columns).index('x')
    y_idx_out = list(df_out.columns).index('y')
    
    for row in df_out.itertuples(index=False, name=None):
        j = ids[row[:3]]
        values = list(id_to_values[j])
        values[x_idx] = x[j][-1]
        values[y_idx] = y[j][-1]
        values.append(values[x_ball_idx]-values[x_idx])
        values.append(values[y_ball_idx]-values[y_idx])
        x[j].append(row[x_idx_out])
        y[j].append(row[y_idx_out])

        if len(x[j])<trace_num+1:
            continue
        for k in range(min(trace_num, len(x[j])-1)):
            values.append(x[j][~k]-x[j][~(k+1)])
            values.append(y[j][~k]-y[j][~(k+1)])
        rows.append(values)

    return rows

In [67]:
train_in, train_out, test_in, test_out = read()

In [68]:
train_in_1, train_out_1, encode_list = f(train_in, train_out)

In [69]:
train_in_1.head(1)

,game_id,play_id,player_to_predict,nfl_id,frame_id,play_direction,absolute_yardline_number,player_name,player_height,player_weight,player_birth_date,player_position,player_side,player_role,x,y,s,a,dir,o,num_frames_output,ball_land_x,ball_land_y
0,2023090700,101,0,54527,1,0,42,0,73,210,799886,0,0,0,52.33,36.94,0.09,0.39,322.4,238.24,21,63.259998,-0.22


In [83]:
base = list(train_in_1.columns)

# cols = ['nfl_id', 'absolute_yardline_number',
#        'player_height', 'player_weight', 'player_birth_date',
#        'player_position', 'x', 'y',
#        'ball_land_x', 'ball_land_y']

cols = ['absolute_yardline_number',
        'player_position', 'player_side', 'player_role',
        'x', 'y',
        'ball_land_x', 'ball_land_y']

cols_add = ['dist_x', 'dist_y']

for i in range(trace_num):
    cols_add.append('dx_{}'.format(i))
    cols_add.append('dy_{}'.format(i))


# cols_test = ['game_id', 'play_id'] + cols
cols_test = ['game_id', 'play_id', 'nfl_id'] + cols
cols_add_test = cols_add[:2] + cols_add[4:]


In [71]:
rows_train = get_all_data(train_in_1, train_out_1)
all_data_train = pd.DataFrame(rows_train, columns=base + cols_add)

print(all_data_train.shape)

(4404615, 37)


In [72]:
train_in_1.head(1)

,game_id,play_id,player_to_predict,nfl_id,frame_id,play_direction,absolute_yardline_number,player_name,player_height,player_weight,player_birth_date,player_position,player_side,player_role,x,y,s,a,dir,o,num_frames_output,ball_land_x,ball_land_y
0,2023090700,101,0,54527,1,0,42,0,73,210,799886,0,0,0,52.33,36.94,0.09,0.39,322.4,238.24,21,63.259998,-0.22


In [73]:
all_data_train.head(1)

,game_id,play_id,player_to_predict,nfl_id,frame_id,play_direction,absolute_yardline_number,player_name,player_height,player_weight,player_birth_date,player_position,player_side,player_role,x,y,s,a,dir,o,num_frames_output,ball_land_x,ball_land_y,dist_x,dist_y,dx_0,dy_0,dx_1,dy_1,dx_2,dy_2,dx_3,dy_3,dx_4,dy_4,dx_5,dy_5
0,2023090700,101,0,54527,1,0,42,0,73,210,799886,0,0,0,52.44,36.88,0.09,0.39,322.4,238.24,21,63.259998,-0.22,10.819998,-37.1,0.07,-0.02,0.07,-0.02,0.02,-0.02,0.02,-0.01,0.0,-0.01,0.0,0.0


In [84]:
import lightgbm as lgb 
from sklearn.metrics import mean_squared_error
from itertools import combinations


mask = np.random.rand(all_data_train.shape[0]) < 0.8

# cols_list = []
# scores_x = []
# scores_y = [] 

X = all_data_train[[col for col in cols+cols_add if col not in {'dx_0', 'dy_0'}]]
y_x = all_data_train['dx_0']
y_y = all_data_train['dy_0']

# if trial:
X_train, y_x_train, y_y_train = X[mask], y_x[mask], y_y[mask]
X_valid, y_x_valid, y_y_valid = X[~mask], y_x[~mask], y_y[~mask]
num_round = 1000

evals_result_x = {}
evals_result_y = {}

callbacks_x = [
    lgb.early_stopping(stopping_rounds=10),    
    lgb.log_evaluation(100),
    lgb.record_evaluation(evals_result_x)
]

callbacks_y = [
    lgb.early_stopping(stopping_rounds=10),    
    lgb.log_evaluation(100),
    lgb.record_evaluation(evals_result_y)
]
# else:
#     X_train, y_x_train, y_y_train = X, y_x, y_y
#     X_valid, y_x_valid, y_y_valid = X.iloc[0:1], y_x.iloc[0:1], y_y.iloc[0:1]
#     X_train, y_x_train, y_y_train = X[mask], y_x[mask], y_y[mask]
#     X_valid, y_x_valid, y_y_valid = X[~mask], y_x[~mask], y_y[~mask]
    
#     num_round = 1000
    
#     evals_result_x = {}
#     evals_result_y = {}
    
#     callbacks_x = [lgb.log_evaluation(100),]
#     callbacks_y = [lgb.log_evaluation(100),]
    

lgb_train_x = lgb.Dataset(X_train, y_x_train)
lgb_eval_x = lgb.Dataset(X_valid, y_x_valid)

lgb_train_y = lgb.Dataset(X_train, y_y_train)
lgb_eval_y = lgb.Dataset(X_valid, y_y_valid)

params = {'objective': 'regression', 'seed': 71, 'verbose': 1, 
         'metrics': 'rmse', 'learning_rate': 0.1, 'num_leaves': 31,
         }


model_x = lgb.train(
    params, 
    train_set = lgb_train_x,
    num_boost_round = num_round,
    valid_sets = [lgb_train_x, lgb_eval_x],
    valid_names = ['train', 'eval'],
    callbacks = callbacks_x,
)

model_y = lgb.train(
    params, 
    train_set = lgb_train_y,
    num_boost_round = num_round,
    valid_sets = [lgb_train_y, lgb_eval_y],
    valid_names = ['train', 'eval'],
    callbacks = callbacks_y,
)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.153997 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4615
[LightGBM] [Info] Number of data points in the train set: 3524140, number of used features: 23
[LightGBM] [Info] Start training from score -0.000019
Training until validation scores don't improve for 10 rounds
[100]	train's rmse: 0.0150631	eval's rmse: 0.0154518
[200]	train's rmse: 0.0146025	eval's rmse: 0.0151158
[300]	train's rmse: 0.0143585	eval's rmse: 0.0149507
[400]	train's rmse: 0.0141833	eval's rmse: 0.0148281
[500]	train's rmse: 0.0140408	eval's rmse: 0.0147331
[600]	train's rmse: 0.0139328	eval's rmse: 0.0146653
[700]	train's rmse: 0.0138395	eval's rmse: 0.0146112
[800]	train's rmse: 0.0137613	eval's rmse: 0.0145618
[900]	train's rmse: 0.0136823	eval's rmse: 0.0145134
[1000]	train's rmse: 0.0136154	eval's rmse: 0.0144757

In [75]:
# plt.figure()
# plt.plot(evals_result_x['train']['rmse'], label='train')
# plt.plot(evals_result_x['eval']['rmse'], label='eval')
# plt.legend()



# plt.figure()
# plt.plot(evals_result_y['train']['rmse'], label='train')
# plt.plot(evals_result_y['eval']['rmse'], label='eval')
# plt.legend()

In [76]:
features = model_x.feature_name()
importance = model_x.feature_importance()
df_importance = pd.DataFrame({
    "feature": features,
    "importance": importance
}).sort_values(by="importance", ascending=False)
df_importance

,feature,importance
10,dx_1,5155
18,dx_5,4877
12,dx_2,4628
14,dx_3,3546
16,dx_4,3483
0,absolute_yardline_number,1037
8,dist_x,847
4,x,807
1,player_position,788
5,y,638


In [77]:
# features = model_y.feature_name()
# importance = model_y.feature_importance()
# df_importance = pd.DataFrame({
#     "feature": features,
#     "importance": importance
# }).sort_values(by="importance", ascending=False)
# df_importance

In [78]:
def predict(df_out, df_in):
    if not isinstance(df_in, pd.DataFrame):
        df_in = df_in.to_pandas()
        df_out = df_out.to_pandas()
    df_in_1, df_out_1, _ = f(df_in, df_out, encode_list)
    df_out_1 = df_out_1.drop(columns=['id'])
    df_in_1 = df_in_1[cols_test]

    ids = {}
    x = []
    y = []
    id_to_values = []
    idx = 0
    

    num_cols = len(cols_test) + len(cols_add_test)
    x_idx = cols_test.index('x')
    y_idx = cols_test.index('y')
    x_ball_idx = cols_test.index('ball_land_x')
    y_ball_idx = cols_test.index('ball_land_y')

    predictions = [[], []]
    
    for row in df_in_1.itertuples(index=False, name=None):
        t = row[:3]
        if t in ids:
            j = ids[t]
        else:
            ids[t] = idx
            j = idx
            idx += 1
            id_to_values.append(row)
            x.append([])
            y.append([])
            
        x[j].append(row[x_idx])
        y[j].append(row[y_idx])


    # tmp = []
    for row in df_out_1.itertuples(index=False, name=None):
        t = row[:3]
        j = ids[t]
        values = list(id_to_values[j])
        values[x_idx] = x[j][-1]
        values[y_idx] = y[j][-1]
        values.append(values[x_ball_idx]-values[x_idx])
        values.append(values[y_ball_idx]-values[y_idx])
        for k in range(min(trace_num-1, len(x[j])-1)):
            values.append(x[j][~k]-x[j][~(k+1)])
            values.append(y[j][~k]-y[j][~(k+1)])
        values += [np.nan]*(num_cols-len(values))
        dx = model_x.predict([values[3:]])[0]
        dy = model_y.predict([values[3:]])[0]
        nx, ny = x[j][-1]+dx, y[j][-1]+dy

        x[j].append(nx)
        y[j].append(ny)
        predictions[0].append(nx)
        predictions[1].append(ny)

        

    submission = pd.DataFrame({'x': predictions[0], 'y': predictions[1]})
    # print(tmp)
    return submission
    

In [79]:
test_in= pd.read_csv(dir / 'test_input.csv')
test_out = pd.read_csv(dir / 'test.csv')
preds = predict(test_out, test_in)
preds

,x,y
0,88.344995,34.294757
1,88.685367,34.310730
2,89.066898,34.377510
3,89.486051,34.499245
4,89.927291,34.673727
...,...,...
5832,98.427642,32.411217
5833,99.096891,33.148828
5834,99.764391,33.885506
5835,100.430218,34.621311


In [80]:
import kaggle_evaluation.nfl_inference_server
import os
inference_server = kaggle_evaluation.nfl_inference_server.NFLInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/nfl-big-data-bowl-2026-prediction/',))
